# Electricity Load Forecasting (h+1) - Real Time Data Ingestion

The aim of this notebok is to understand the **real time data ingestion pipeline** needed to build features and run an h+1 prediction. 

## 0. Setup

In [1]:
import sys
import os
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# Add the project root, api and src directories to the Python path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "api"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "ingestion"))

print(f"Project root : {PROJECT_ROOT}")

Project root : /Users/bachirijihane/energy-intelligence-platform


In [2]:
from src.ingestion import get_realtime_data

## 1. Fetch ENTSO-E Real-Time Demand Data

In [3]:
# Load environment variable (ENTSOE_API_TOKEN) from .env file
load_dotenv()
entsoe_api_token = os.getenv("ENTSOE_API_TOKEN")

In [4]:
df_entsoe = get_realtime_data.fetch_entsoe_realtime(
    country_code="10YFR-RTE------C",
    api_token=entsoe_api_token,
    lookback_hours=48,
)

[ENTSOE] Fetching demand from 202604051427 to 202604071427 UTC
[DEBUG] Before resample : 54 lines
[DEBUG] After resample : 48 lines
[ENTSOE] Fetched 48 hourly rows


In [5]:
print("Head:\n", df_entsoe.head())
print("Tail:\n", df_entsoe.tail())

Head:
                    datetime   load_MW
0 2026-04-05 15:00:00+00:00  38227.65
1 2026-04-05 16:00:00+00:00  38107.31
2 2026-04-05 17:00:00+00:00  38256.15
3 2026-04-05 18:00:00+00:00  38276.61
4 2026-04-05 19:00:00+00:00  38173.53
Tail:
                     datetime    load_MW
43 2026-04-07 10:00:00+00:00  44266.840
44 2026-04-07 11:00:00+00:00  44282.325
45 2026-04-07 12:00:00+00:00  44125.140
46 2026-04-07 13:00:00+00:00  44400.350
47 2026-04-07 14:00:00+00:00  44199.160


In [6]:
df_entsoe.info(), df_entsoe.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype              
---  ------    --------------  -----              
 0   datetime  48 non-null     datetime64[ns, UTC]
 1   load_MW   48 non-null     float64            
dtypes: datetime64[ns, UTC](1), float64(1)
memory usage: 900.0 bytes


(None,
             load_MW
 count     48.000000
 mean   43056.110938
 std     2545.227294
 min    38107.310000
 25%    41532.350000
 50%    44040.330000
 75%    44851.527500
 max    46260.010000)

## 2. Fetch Open-Meteo Real-Time Weather Data

In [7]:
df_weather = get_realtime_data.fetch_openmeteo_realtime(
    latitude=48.8534, # Paris coordinates
    longitude=2.3488,
    forecast_hours=2 # short-term forecast for the next 2 hours
)

[OPENMETEO] Fetching weather forecast for lat=48.8534, lon=2.3488
[OPENMETEO] Fetched 2 forecast rows


In [8]:
print("Weather Data:\n", df_weather)

Weather Data:
                    datetime  temperature_2m  relative_humidity_2m  \
0 2026-04-07 14:00:00+00:00       24.369501                  34.0   
1 2026-04-07 15:00:00+00:00       24.619501                  35.0   

   wind_speed_10m  shortwave_radiation_instant  
0       11.384199                   657.807373  
1       10.948973                   534.740540  


## 3. Build a Real-Time Snapshot by Merging Demand and Weather DataFrames

In [9]:
df_merged = get_realtime_data.build_realtime_snapshot(df_entsoe, df_weather)

In [10]:
df_merged.tail()

,datetime,load_MW,temperature_2m,relative_humidity_2m,wind_speed_10m,shortwave_radiation_instant
43,2026-04-07 10:00:00+00:00,44266.840,NaN,NaN,NaN,NaN
44,2026-04-07 11:00:00+00:00,44282.325,NaN,NaN,NaN,NaN
45,2026-04-07 12:00:00+00:00,44125.140,NaN,NaN,NaN,NaN
46,2026-04-07 13:00:00+00:00,44400.350,24.369501,34.0,11.384199,657.807373
47,2026-04-07 14:00:00+00:00,44199.160,24.619501,35.0,10.948973,534.740540


The Open-Meteo `/forecast` endpoint returns only *t* and *t+1* (2 rows). After applying the 1-hour shift in `build_realtime_snapshot`, these two rows are merged with the demand data at *t-1* and *t*.

All other demand rows (48-hour history) have `NaN` values for weather, which is expected, since `realtime.parquet` is not meant to store historical weather data. This file will be completed with historical data from `data/processed/` when features are built within the API.

We can also confirm that the shift is working correctly, as the weather data is properly aligned at *t-1*.

## 4. Save the Real-Time Snapshot 

In [11]:
get_realtime_data.fetch_and_store_realtime(
    country="FR",
    country_code="10YFR-RTE------C",
    latitude=48.8534,
    longitude=2.3488,
)


[REALTIME] Starting real-time ingestion | country=FR
[REALTIME] Timestamp: 2026-04-07 14:27 UTC
[ENTSOE] Fetching demand from 202604051427 to 202604071427 UTC
[DEBUG] Before resample : 54 lines
[DEBUG] After resample : 48 lines
[ENTSOE] Fetched 48 hourly rows
[OPENMETEO] Fetching weather forecast for lat=48.8534, lon=2.3488
[OPENMETEO] Fetched 2 forecast rows
[REALTIME] Snapshot built | rows=48
[SAVED] /Users/bachirijihane/energy-intelligence-platform/data/realtime/country=FR/realtime.parquet | total rows in window = 48
[REALTIME] Done.


In [16]:
# Read the stored parquet file to verify it was saved correctly
df_real_time = pd.read_parquet(PROJECT_ROOT / "data" / "realtime" / "country=FR" /"realtime.parquet")
df_real_time.tail(), df_real_time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   datetime                     48 non-null     datetime64[ns, UTC]
 1   load_MW                      48 non-null     float64            
 2   temperature_2m               2 non-null      float32            
 3   relative_humidity_2m         2 non-null      float32            
 4   wind_speed_10m               2 non-null      float32            
 5   shortwave_radiation_instant  2 non-null      float32            
 6   country                      48 non-null     object             
dtypes: datetime64[ns, UTC](1), float32(4), float64(1), object(1)
memory usage: 2.0+ KB


(                    datetime    load_MW  temperature_2m  relative_humidity_2m  \
 43 2026-04-07 10:00:00+00:00  44266.840             NaN                   NaN   
 44 2026-04-07 11:00:00+00:00  44282.325             NaN                   NaN   
 45 2026-04-07 12:00:00+00:00  44125.140             NaN                   NaN   
 46 2026-04-07 13:00:00+00:00  44400.350       24.369501                  34.0   
 47 2026-04-07 14:00:00+00:00  44199.160       24.619501                  35.0   
 
     wind_speed_10m  shortwave_radiation_instant country  
 43             NaN                          NaN      FR  
 44             NaN                          NaN      FR  
 45             NaN                          NaN      FR  
 46       11.384199                   657.807373      FR  
 47       10.948973                   534.740540      FR  ,
 None)

The file is identical to the one obtained from the merge, the real-time ingestion pipeline is working as expected.